Connected to .venv (Python 3.10.11)

In [ ]:
from datasets import load_dataset

ds = load_dataset("Marqo/polyvore")


In [ ]:
data = ds["data"]

print(len(data))

94096


In [ ]:
set(data["category"])

{'Accent Chairs',
 'Accent Tables',
 'Accessories',
 'Activewear',
 'Activewear Jackets',
 'Activewear Pants',
 'Activewear Shorts',
 'Activewear Skirts',
 'Activewear Tank Tops',
 'Activewear Tops',
 'Ankle Booties',
 'Aprons',
 'Armoires',
 'Athletic Shoes',
 'Baby',
 'Baby Bedding',
 'Backpacks',
 'Bags',
 'Bags & Cases',
 'Bakeware',
 'Bar Cabinets',
 'Bar Tools',
 'Barstools',
 'Bath',
 'Bath & Body',
 'Bath Accessories',
 'Bath Rugs & Mats',
 'Bath Towels',
 'Beach Towels',
 'Beauty Accessories',
 'Beauty Products',
 'Bed Accessories',
 'Bed Pillows',
 'Bedding',
 'Beds',
 'Bedspreads',
 'Belts',
 'Benches',
 'Bikini Bottoms',
 'Bikini Tops',
 'Bikinis',
 'Blankets',
 'Blazers',
 'Blouses',
 'Blow Dryers & Irons',
 'Blush',
 'Body Art',
 'Body Cleansers',
 'Body Moisturizers',
 'Bookcases',
 'Books',
 'Bootcut Jeans',
 'Boots',
 'Bow Ties',
 'Boyfriend Jeans',
 'Boys',
 'Bracelets & Bangles',
 'Bras',
 'Briefcases',
 'Brooches',
 'Brushes & Combs',
 'Cabinets',
 'Camisoles',
 'Ca

In [ ]:
CLOTHING_MAP = {

    # -------- TOPS --------
    "Blouses": "tops",
    "T-Shirts": "tops",
    "Tank Tops": "tops",
    "Sweaters": "tops",
    "Cardigans": "tops",
    "Tunics": "tops",
    "Hoodies": "tops",
    "Tops": "tops",

    # -------- BOTTOMS --------
    "Jeans": "bottoms",
    "Skinny Jeans": "bottoms",
    "Bootcut Jeans": "bottoms",
    "Flared Jeans": "bottoms",
    "Wide Leg Jeans": "bottoms",
    "Pants": "bottoms",
    "Shorts": "bottoms",
    "Skirts": "bottoms",
    "Mini Skirts": "bottoms",
    "Long Skirts": "bottoms",

    # -------- DRESSES --------
    "Day Dresses": "dresses",
    "Cocktail Dresses": "dresses",
    "Dresses": "dresses",
    "Wedding Dresses": "dresses",
    "Gowns": "dresses",

    # -------- OUTERWEAR --------
    "Jackets": "outerwear",
    "Coats": "outerwear",
    "Blazers": "outerwear",
    "Outerwear": "outerwear",
}

In [ ]:
def keep_clothes(example):
    cat = example["category"]

    if cat in CLOTHING_MAP:
        example["norm_category"] = CLOTHING_MAP[cat]
        return example

    example["norm_category"] = None
    return example


data = data.map(keep_clothes)

Map: 100%|██████████| 94096/94096 [00:15<00:00, 6092.24 examples/s] 


In [ ]:
data = data.filter(lambda x: x["norm_category"] is not None)

Filter: 100%|██████████| 94096/94096 [01:43<00:00, 913.05 examples/s] 


In [ ]:
set(data["norm_category"])

{'bottoms', 'dresses', 'outerwear', 'tops'}

In [ ]:
data = data.filter(
    lambda x: x["norm_category"] in ["tops", "bottoms"]
)

Filter: 100%|██████████| 25093/25093 [00:22<00:00, 1108.93 examples/s]


In [ ]:
set(data["norm_category"])

{'bottoms', 'tops'}

In [ ]:
from collections import Counter

Counter(data["norm_category"])

Counter({'tops': 9871, 'bottoms': 6013})

In [ ]:
print(data[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=145x400 at 0x2849EFFB5E0>, 'category': 'Skinny Jeans', 'text': 'givenchy skinny jean', 'item_ID': '100010727_2', 'norm_category': 'bottoms'}


In [ ]:
data = data.shuffle(seed=42)

In [ ]:
train_test = data.train_test_split(test_size=0.3, seed=42)

test_valid = train_test["test"].train_test_split(test_size=0.5, seed=42)

train_data = train_test["train"]
valid_data = test_valid["train"]
test_data = test_valid["test"]

In [ ]:
print(len(train_data))
print(len(valid_data))
print(len(test_data))

11118
2383
2383


In [ ]:
train_tops = train_data.filter(lambda x: x["norm_category"] == "tops")
train_bottoms = train_data.filter(lambda x: x["norm_category"] == "bottoms")

Filter: 100%|██████████| 11118/11118 [00:10<00:00, 1065.78 examples/s]


In [ ]:
test_tops = test_data.filter(lambda x: x["norm_category"] == "tops")
test_bottoms = test_data.filter(lambda x: x["norm_category"] == "bottoms")

Filter: 100%|██████████| 2383/2383 [00:02<00:00, 814.92 examples/s]


In [ ]:
valid_tops = valid_data.filter(lambda x: x["norm_category"] == "tops")
valid_bottoms = valid_data.filter(lambda x: x["norm_category"] == "bottoms")

Filter: 100%|██████████| 2383/2383 [00:02<00:00, 1077.77 examples/s]


In [ ]:
train_top = list(train_tops)
train_bottom = list(train_bottoms)

In [ ]:
import random

def create_pairs(tops, bottoms, num_pairs=None):
    pairs = []

    if num_pairs is None:
        num_pairs = min(len(tops), len(bottoms))

    for _ in range(num_pairs):

        # ---------- POSITIVE PAIR ----------
        top = random.choice(tops)
        bottom = random.choice(bottoms)

        pairs.append({
            "top_image": top["image"],
            "bottom_image": bottom["image"],
            "label": 1
        })

        # ---------- NEGATIVE PAIR ----------
        top_neg = random.choice(tops)
        bottom_neg = random.choice(bottoms)

        pairs.append({
            "top_image": top_neg["image"],
            "bottom_image": bottom_neg["image"],
            "label": 0
        })

    return pairs

In [ ]:
train_pairs = create_pairs(train_top, train_bottom, num_pairs=10000)

In [ ]:
test_top = list(test_tops)
test_bottom = list(test_bottoms)



In [ ]:
valid_top = list(valid_tops)
valid_bottom = list(valid_bottoms)

In [ ]:
valid_pairs = create_pairs(valid_top, valid_bottom, num_pairs=2000)

In [ ]:
test_pairs = create_pairs(test_top, test_bottom, num_pairs=2000)

In [ ]:
train_pairs[0]

{'top_image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=355x400>,
 'bottom_image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=344x400>,
 'label': 1}